In [2]:
import pandas as pd

In [51]:

import textwrap
import sys 
sys.path.append('../../')
import utils.llm_training as llm_training 

import utils.utils as utils
import pandas as pd

with open('../../data/arxiv/cleaned_DPO.txt', 'r') as f:
    paper = f.read()


def print_wrapped(text, width=100):
    """
    Prints the given text wrapped to a specified width for better readability in notebooks.
    This function preserves paragraph breaks.
    """
    paragraphs = text.split('\n\n')
    for para in paragraphs:
        print(textwrap.fill(para, width=width))
        print()
        
print_wrapped(paper)



\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract} While large-scale unsupervised language models (LMs) learn broad world knowledge
and some reasoning skills, achieving precise control of their behavior is difficult due to the
completely unsupervised nature of their training. Existing methods for gaining such steerability
collect human labels of the relative quality of model generations and fine-tune the unsupervised LM
to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects
the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning
to maximize this estimated reward without drifting too far from the original model. In this paper we
introduce a new parameterization of the reward model in RLHF that enables extraction of the
corresponding optimal policy in clo

### 1. Generate High-Level Questions

In [40]:
# Parse paper into sections
# sections_df = parse_paper_structure(paper)
from importlib import reload
reload(utils)
# Create questions for each section
all_questions = []

# # Get the full text for this section
# section_text = section_group['section_text'].iloc[0]

#Then, at the end, turn these questions into cloze form sentences in which the answer is placed at the end of the sentence. Provide the cloze sentence with the answer missing, and the answer in JSON.

section_text = paper
prompt = {}
prompt['system'] = """### Instructions
Based on your understanding of the provided text, generate inference questions to test a reader's comprehension and understanding of the text. The most important aspect is that the question should build upon the text to draw a conclusion or synthesize the knowledge.
- The question can be as long as needed, but the answer should be a phrase that is 1-5 words. 
- The questions should NOT be about factual recall that asks to recall a specific fact verbatim. 
- The answer to the question should not be found in the text alone. It should be a conclusion or synthesis of the knowledge.
- The questions should require a generalizable, deep understanding of the knowledge. 
- Be precise with the question formulation so that there is only one clear answer.

In addition, for each question, provide: 
- The prior knowledge that is required to answer the question. Academic papers build upon a large body of domain knowledge, and so there is an underlying assumption that the reader has a deep understanding of the domain knowledge for any paper.
- The sentences from the text that are required to answer the question. Cite from the text verbatim, and don't surround it with quotes.
- For a lay person, an explanation of what the question is asking.
- For a lay person, an explanation of how the answer is derived from the provided knowledge in the text.

### Output Format
Provide the question and answers in JSON, as a list of dictionaries with the following keys:
- "question": (string) 
- "answer": (string)
- "prior_knowledge": (string)
- "text_sentences": list of strings
- "question_explanation": (string)
- "inference_explanation": (string)
"""

prompt['user'] = f"""### Text
{section_text}"""

# Query LLM for questions
response = utils.query_llm(prompt, model='gpt-5', system_prompt_included=True, return_json=True, max_tokens=1000)

# Parse the JSON response
import json
parsed_questions = []

# Handle string response that needs JSON parsing
if isinstance(response, str):
    try:
        response = json.loads(response)
    except json.JSONDecodeError:
        print("Failed to parse JSON response")
        response = []


In [43]:
print(response)

{'qa_items': [{'question': 'Interpreting the optimal policy expression π_r(y|x) ∝ p_ref(y|x) · exp(r/β) as a Bayesian-style update, in what probabilistic role does the reference policy p_ref function within DPO’s learning view?', 'answer': 'a prior distribution', 'prior_knowledge': "Understanding that in Bayesian inference the posterior is proportional to the prior times a likelihood term; recognizing exp(reward/temperature) as an exponential family 'likelihood' factor that tilts a prior distribution.", 'text_sentences': ['it is straightforward to show that the optimal solution to the KL-constrained reward maximization objective in Eq.~\\ref{eq:RL} takes the form:', 'where $Z(x) =\\sum_{y}\\piref(y\\mid x)\\exp\\left(\\frac{1}{\\beta}r(x, y)\\right)$ is the partition function.', 'The added constraint is important, as it prevents the model from deviating too far from the distribution on which the reward model is accurate, as well as maintaining the generation diversity and preventing mo

In [59]:
# Extract questions from response
dropped = 0
if isinstance(response, dict) and 'qa_items' in response:
    for item in response['qa_items']:
        # Check if all text sentences are actually in the paper
        all_sentences_found = True
        if 'text_sentences' in item and isinstance(item['text_sentences'], list):
            for sentence in item['text_sentences']:
                if sentence not in section_text:
                    all_sentences_found = False
                    break
        
        # Only process if all sentences are found in the paper
        if all_sentences_found:
            print_wrapped(f"Question: {item['question']}")
            print_wrapped(f"Answer: {item['answer']}")
            print_wrapped(f"Prior Knowledge: {item['prior_knowledge']}")
            print_wrapped(f"Text Sentences: {item['text_sentences']}")
            print_wrapped(f"Question Explanation: {item['question_explanation']}")
            print_wrapped(f"Inference Explanation: {item['inference_explanation']}")
            print_wrapped("---")
            
            parsed_questions.append({
                'question': item['question'],
                'answer': item['answer'],
                'prior_knowledge': item['prior_knowledge'],
                'text_sentences': item['text_sentences'],
                'question_explanation': item['question_explanation'],
                'inference_explanation': item['inference_explanation']
            })
        else:
            print_wrapped(f"DROPPED: Question with invalid text sentences: {item['question']}")
            dropped += 1

print(f"Dropped {dropped} out of {len(response['qa_items'])} questions")


Question: Interpreting the optimal policy expression π_r(y|x) ∝ p_ref(y|x) · exp(r/β) as a Bayesian-
style update, in what probabilistic role does the reference policy p_ref function within DPO’s
learning view?

Answer: a prior distribution

Prior Knowledge: Understanding that in Bayesian inference the posterior is proportional to the prior
times a likelihood term; recognizing exp(reward/temperature) as an exponential family 'likelihood'
factor that tilts a prior distribution.

Text Sentences: ['it is straightforward to show that the optimal solution to the KL-constrained
reward maximization objective in Eq.~\\ref{eq:RL} takes the form:', 'where $Z(x)
=\\sum_{y}\\piref(y\\mid x)\\exp\\left(\\frac{1}{\\beta}r(x, y)\\right)$ is the partition
function.', 'The added constraint is important, as it prevents the model from deviating too far from
the distribution on which the reward model is accurate, as well as maintaining the generation
diversity and preventing mode-collapse to single high-r

#### 1.1 Convert Questions into Cloze Form Sentences

In [56]:
# Convert questions to text format with embedded answers using LLM
cloze_prompt = {}
cloze_prompt['system'] = r"""### Instructions
You will be given a list of pairs of questions and answers. Turn each question and answer into a statement in which the answer is placed at the very end. 
- The answer must be at the *very end* of the the last sentence. There should be no other words after the answer.
- The paraphrasing should be natural and flow well, with the answer naturally appearing at the very end. Avoid awkward repetition or forced placement of the answer. If restructuring the question to end with the answer feels unnatural or forced, discard that question entirely.
- Ensure the statement is grammatically correct and well-written. Adjust the answer only if necessary to maintain grammatical correctness.
- Preserve all information and context from the original question when converting to cloze form.
- Maintain the same sentence count as the original question.

### Demonstration

Question: Within the DPO theoretical section, the projection operator f normalizes the reward function by subtracting a term involving the policy’s partition function. Which mathematical function of the partition function is used for this normalization?  
Answer: logarithm

Statement: Within the DPO theoretical section, the projection operator f normalizes the reward function by subtracting a term involving the policy’s partition function. The mathematical function of the partition function used for this normalization is the logarithm.

### Output Format 
List the statements as a list of (answer, statement)."""
cloze_prompt['user'] = f"""### Questions and Answers\n{json.dumps(response['qa_items'])}\n"""
converted_response = utils.query_llm(cloze_prompt, model='gpt-5-mini', system_prompt_included=True, max_tokens=10000)

quality_control_prompt = {}
quality_control_prompt['system'] = r"""You are a meticulous Quality Control Assistant. Your task is to review and refine statements that have been extracted from an academic paper. You will be given a list of '(answer, statement)' pairs and the original context from which they were derived. Your task is to apply a rigorous checklist to each pair, refining it based on a provided quality control checklist.

### Quality Control Checklist

For each '(answer, statement)' pair, verify the following:

1. LaTeX Formatting
- All mathematical expressions and notations **MUST** be written in LaTeX, enclosed in '$' or '$$' delimiters.
- Do *NOT* use unicode mathematical characters (e.g., use '\\pi', not 'π').
- Do *NOT* use unnecessary styling commands like '\\displaystyle'.
- Ensure LaTeX syntax matches the style of the original context (e.g., '( ... )' or '$ ... $').
- Action: Rewrite the math expressions and statements so they can be written in LaTeX, keeping the rest of the statement the same, correcting any and all formatting errors related to mathematical notation.

In all your adjustments, change the statement as minimally as necessary. If a statement is already good, make no changes.
### Output Format
After your review, provide a list of the refined '(answer, statement)' pairs that have passed all checks in JSON."""
quality_control_prompt['user'] = f"""### Questions and Answers\n{converted_response}"""
refined_converted_response = utils.query_llm(quality_control_prompt, model='gpt-5-mini', system_prompt_included=True, return_json=True, max_tokens=10000)
refined_converted_response = json.loads(refined_converted_response)
print(refined_converted_response)


{'pairs': [['a prior distribution', "Interpreting the optimal policy expression $\\pi_r(y\\mid x) \\propto p_{\\text{ref}}(y\\mid x)\\exp(r/\\beta)$ as a Bayesian-style update in DPO's learning view, the reference policy $p_{\\text{ref}}$ plays the role of a prior distribution."], ['partition function cancellation', "If the preference model depended on absolute reward levels rather than only on reward differences between responses, the critical algebraic simplification that enables DPO's closed-form policy learning would fail: partition function cancellation."], ['the soft value function', "In actor-critic terms and relative to PPO, DPO's reparameterization removes the need to estimate the soft value function."], ['the optimal policy', 'Given that adding any function $f(x)$ to the reward leaves Bradley-Terry/Plackett-Luce preferences unchanged, DPO still uniquely recovers the optimal policy.'], ['small update', "When the implicit reward already orders the preferred response above the d

#### 1.2 Split the sentences into "probe" and "target"

This is for evaluating the prediction of the target, given the rpobe

In [61]:
# Create probes from the refined response and merge with original QA data
valid_probes = []
dropped_count = 0

for i, (answer, statement) in enumerate(refined_converted_response['pairs']):
    # Strip punctuation from the end of the statement and check if it ends with the answer
    statement_stripped = statement.rstrip('.,!?;: ')
    if statement_stripped.lower().endswith(answer.lower()):
        # Create probe by removing the answer from the end
        probe = statement_stripped[:-len(answer)].rstrip()
        
        # Merge with original QA data
        original_qa = parsed_questions[i] if i < len(parsed_questions) else {}
        valid_probes.append({
            'target': answer,
            'probe': probe,
            'fact': statement,
            'question': original_qa.get('question', ''),
            'answer': original_qa.get('answer', ''),
            'prior_knowledge': original_qa.get('prior_knowledge', ''),
            'text_sentences': original_qa.get('text_sentences', ''),
            'question_explanation': original_qa.get('question_explanation', ''),
            'inference_explanation': original_qa.get('inference_explanation', '')
        })
    else:
        dropped_count += 1

print(f"Dropped {dropped_count} statements that don't end with their answer")


Dropped 0 statements that don't end with their answer


In [ ]:
# Create DataFrame and save as CSV
df = pd.DataFrame(valid_probes)
df.to_csv('dpo_high_level_probes_v2.csv', index=False)
print(f"Created {len(valid_probes)} probes and saved to dpo_high_level_probes_formatted.csv")
df

Created 12 probes and saved to dpo_high_level_probes_formatted.csv


,target,probe,fact,question,answer,prior_knowledge,text_sentences,question_explanation,inference_explanation
0,a prior distribution,Interpreting the optimal policy expression $\p...,Interpreting the optimal policy expression $\p...,Interpreting the optimal policy expression π_r...,a prior distribution,Understanding that in Bayesian inference the p...,[it is straightforward to show that the optima...,It asks you to see the reference model’s role ...,Because the optimal policy is proportional to ...
1,partition function cancellation,If the preference model depended on absolute r...,If the preference model depended on absolute r...,If the preference model depended on absolute r...,partition function cancellation,Knowing that subtracting or differencing remov...,"[Fortunately, the Bradley-Terry model depends ...",It asks which mathematical trick DPO relies on...,DPO leverages that preferences depend on rewar...
2,the soft value function,"In actor-critic terms and relative to PPO, DPO...","In actor-critic terms and relative to PPO, DPO...","In actor–critic terms, which quantity does DPO...",the soft value function,Actor–critic methods reduce gradient variance ...,[We can also use our framework to diagnose ins...,It asks what extra learned component PPO needs...,Since the normalization term acts as a soft va...
3,the optimal policy,Given that adding any function $f(x)$ to the r...,Given that adding any function $f(x)$ to the r...,Given that adding any function f(x) to the rew...,the optimal policy,Equivalence classes: many reward functions can...,"[Under the Plackett-Luce, and in particular th...",It asks what end result is still uniquely dete...,Because all rewards in an equivalence class in...
4,small update,When the implicit reward already orders the pr...,When the implicit reward already orders the pr...,When the implicit reward already orders the pr...,small update,The logistic function σ(z) approaches 0 when z...,[The gradient with respect to the parameters $...,It asks what happens to the training signal wh...,"Because the weight is σ(r_l − r_w), and r_w ≫ ..."
5,on-policy sampling,By turning preference learning into supervised...,By turning preference learning into supervised...,What expensive training-loop component of PPO-...,on-policy sampling,PPO training is on-policy and repeatedly sampl...,"[The resulting algorithm, which we call \texti...",It asks which costly PPO step DPO removes by f...,Because DPO uses a fixed preference dataset an...
6,high-variance gradients,Because DPO and PPO optimize the same constrai...,Because DPO and PPO optimize the same constrai...,Because DPO and PPO optimize the same constrai...,high-variance gradients,"When two methods optimize the same objective, ...",[We find that DPO produces by far the most eff...,It asks you to infer what kind of training ins...,"Since objectives match, PPO’s inferiority poin..."
7,mode collapse,"In language generation, the KL constraint in t...","In language generation, the KL constraint in t...","In language generation, what failure mode is t...",mode collapse,Mode collapse is when a model outputs a narrow...,"[The added constraint is important, as it prev...",It asks what diversity-related problem the KL ...,The text states the KL prevents mode-collapse ...
8,a likelihood ratio,By defining the implicit reward as $\beta\log\...,By defining the implicit reward as $\beta\log\...,By defining the implicit reward as β·log(π/ p_...,a likelihood ratio,A likelihood (or density) ratio π/p_ref reweig...,"[Specifically, we first take the logarithm of ...",It asks what mathematical factor transforms th...,"Because the implicit reward is β·log(π/p_ref),..."
9,valid probability distribution,"Under DPO's reparameterization, the learned po...","Under DPO's reparameterization, the learned po...","Under DPO’s reparameterization, what condition...",valid probability distribution,A probability distribution must have nonnegati...,[We can alternatively view Theorem~\ref{thm:ma...,It 

#### 1.3 Clean the probes and targets

In [65]:
df = pd.read_csv('../../data/arxiv/DPO_high_level_probes_v2.csv')


In [66]:
# Clean up the data: strip whitespace, remove punctuation, add space to target
df_cleaned = df.copy()

for i in range(len(df_cleaned)):
    # Strip whitespace from probe, target, and fact
    df_cleaned.at[i, 'probe'] = df_cleaned.at[i, 'probe'].strip()
    df_cleaned.at[i, 'target'] = df_cleaned.at[i, 'target'].strip()
    df_cleaned.at[i, 'fact'] = df_cleaned.at[i, 'fact'].strip()
    
    # Remove punctuation from end of target and fact if present
    target = df_cleaned.at[i, 'target'].rstrip('.,!?;:')
    fact = df_cleaned.at[i, 'fact'].rstrip('.,!?;:')
    
    # Add space in front of target
    target = ' ' + target
    
    # Update the dataframe
    df_cleaned.at[i, 'target'] = target
    df_cleaned.at[i, 'fact'] = fact

# Update df to use the cleaned version
df = df_cleaned
print(f"Cleaned {len(df)} rows: stripped whitespace, removed punctuation, added space to targets")



Cleaned 12 rows: stripped whitespace, removed punctuation, added space to targets


In [68]:
df.to_csv('dpo_high_level_probes_v2.csv', index=False)
print(f"Created {len(valid_probes)} probes and saved to dpo_high_level_probes_formatted.csv")
df

Created 12 probes and saved to dpo_high_level_probes_formatted.csv


,target,probe,fact,question,answer,prior_knowledge,text_sentences,question_explanation,inference_explanation
0,a prior distribution,Interpreting the optimal policy expression $\p...,Interpreting the optimal policy expression $\p...,Interpreting the optimal policy expression π_r...,a prior distribution,Understanding that in Bayesian inference the p...,['it is straightforward to show that the optim...,It asks you to see the reference model’s role ...,Because the optimal policy is proportional to ...
1,partition function cancellation,If the preference model depended on absolute r...,If the preference model depended on absolute r...,If the preference model depended on absolute r...,partition function cancellation,Knowing that subtracting or differencing remov...,"['Fortunately, the Bradley-Terry model depends...",It asks which mathematical trick DPO relies on...,DPO leverages that preferences depend on rewar...
2,the soft value function,"In actor-critic terms and relative to PPO, DPO...","In actor-critic terms and relative to PPO, DPO...","In actor–critic terms, which quantity does DPO...",the soft value function,Actor–critic methods reduce gradient variance ...,['We can also use our framework to diagnose in...,It asks what extra learned component PPO needs...,Since the normalization term acts as a soft va...
3,the optimal policy,Given that adding any function $f(x)$ to the r...,Given that adding any function $f(x)$ to the r...,Given that adding any function f(x) to the rew...,the optimal policy,Equivalence classes: many reward functions can...,"['Under the Plackett-Luce, and in particular t...",It asks what end result is still uniquely dete...,Because all rewards in an equivalence class in...
4,small update,When the implicit reward already orders the pr...,When the implicit reward already orders the pr...,When the implicit reward already orders the pr...,small update,The logistic function σ(z) approaches 0 when z...,['The gradient with respect to the parameters ...,It asks what happens to the training signal wh...,"Because the weight is σ(r_l − r_w), and r_w ≫ ..."
5,on-policy sampling,By turning preference learning into supervised...,By turning preference learning into supervised...,What expensive training-loop component of PPO-...,on-policy sampling,PPO training is on-policy and repeatedly sampl...,"['The resulting algorithm, which we call \\tex...",It asks which costly PPO step DPO removes by f...,Because DPO uses a fixed preference dataset an...
6,high-variance gradients,Because DPO and PPO optimize the same constrai...,Because DPO and PPO optimize the same constrai...,Because DPO and PPO optimize the same constrai...,high-variance gradients,"When two methods optimize the same objective, ...",['We find that DPO produces by far the most ef...,It asks you to infer what kind of training ins...,"Since objectives match, PPO’s inferiority poin..."
7,mode collapse,"In language generation, the KL constraint in t...","In language generation, the KL constraint in t...","In language generation, what failure mode is t...",mode collapse,Mode collapse is when a model outputs a narrow...,"['The added constraint is important, as it pre...",It asks what diversity-related problem the KL ...,The text states the KL prevents mode-collapse ...
8,a likelihood ratio,By defining the implicit reward as $\beta\log\...,By defining the implicit reward as $\beta\log\...,By defining the implicit reward as β·log(π/ p_...,a likelihood ratio,A likelihood (or density) ratio π/p_ref reweig...,"['Specifically, we first take the logarithm of...",It asks what mathematical factor transforms th...,"Because the implicit reward is β·log(π/p_ref),..."
9,valid probability distribution,"Under DPO's reparameterization, the learned po...","Under DPO's reparameterization, the learned po...","Under DPO’s reparameterization, what condition...",valid probability distribution,A probability distribution must have nonnegati...,['We can alternatively view Theorem~\\ref{thm:...,It 

#### 1.4 Split the questions as well actually into "probe" and "target"

We'll be doing the same we did for the cloze form sentences. We also keep the question form to evaluate the model *after* instruction-tuning

In [13]:
import pandas as pd
df = pd.read_csv('../../data/arxiv/DPO_high_level_probes_v2.csv')


In [14]:
# Create probe column by stripping question and adding "\nResponse:"
df['probe'] = df['question'].str.strip() + "\nResponse:"

# Create answer column by stripping answer and adding space before
df['answer'] = ' ' + df['answer'].str.strip()

# Create question_and_answer column by combining probe and answer
df['question_and_answer'] = df['probe'] + df['answer']

df['answer'][0]

' a prior distribution'

In [15]:
df = df.drop(columns=['target', 'fact'])
df.columns

Index(['probe', 'question', 'answer', 'prior_knowledge', 'text_sentences',
       'question_explanation', 'inference_explanation', 'question_and_answer'],
      dtype='object')

In [16]:
df.to_csv('dpo_high_level_probes_in_question_form_v2.csv', index=False)


### 2. Examine Probes

In [ ]:
df = pd.read_csv('../../data/arxiv/DPO_high_level_probes_v2.csv')


In [67]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B")

contexts = df['probe'].tolist()
targets = df['target'].tolist()
facts = df['fact'].tolist()

print(f"--- Tokenizer and String Consistency Check ---")

string_mismatches = 0
token_mismatches = 0

for i, (context, target, fact) in enumerate(zip(contexts, targets, facts)):
    # --- String Level Check ---
    reconstructed_string = context + target
    string_match = reconstructed_string == fact
    
    if not string_match:
        string_mismatches += 1
        print(f"--- ❌ STRING MISMATCH: Sample #{i} ---")
        print(f"Context: '{context}'")
        print(f"Target:  '{target}'")
        print(f"Fact:    '{fact}'")
        print(f"Reconstructed: '{reconstructed_string}'")
        print(f"Match: {string_match}")
        print("-" * 30)
        continue
    
    # --- Tokenization Level Check ---
    # Method 1: Tokenize the fact directly
    tokenized_fact = tokenizer(fact, add_special_tokens=False, padding=False)['input_ids']
    
    # Method 2: Tokenize parts separately and concatenate
    tokenized_context = tokenizer(context, add_special_tokens=False, padding=False)['input_ids']
    tokenized_target = tokenizer(target, add_special_tokens=False, padding=False)['input_ids']
    tokenized_parts_combined = tokenized_context + tokenized_target

    # --- Comparison ---
    if tokenized_fact != tokenized_parts_combined:
        token_mismatches += 1
        print(f"--- ❌ TOKEN MISMATCH: Sample #{i} ---")
        print(f"Context: '{context}'")
        print(f"Target:  '{target}'")
        print(f"Fact:    '{fact}'")
        print(f"\nTokenizing fact directly:           {tokenized_fact} (Length: {len(tokenized_fact)})")
        print(f"Tokenizing parts and concatenating: {tokenized_parts_combined} (Length: {len(tokenized_parts_combined)})")
        
        # Find differing positions
        min_len = min(len(tokenized_fact), len(tokenized_parts_combined))
        diff_positions = []
        for pos in range(min_len):
            if tokenized_fact[pos] != tokenized_parts_combined[pos]:
                diff_positions.append(pos)
        
        if diff_positions:
            print(f"\nFirst differing positions: {diff_positions[:5]}")
            for pos in diff_positions[:3]:
                fact_token = tokenizer.decode([tokenized_fact[pos]])
                parts_token = tokenizer.decode([tokenized_parts_combined[pos]])
                print(f"  Position {pos}: fact='{fact_token}' vs parts='{parts_token}'")
        
        print("-" * 30)

print(f"\nSummary:")
print(f"String mismatches: {string_mismatches}")
print(f"Token mismatches: {token_mismatches}")
print(f"Total samples: {len(contexts)}")

--- Tokenizer and String Consistency Check ---

Summary:
String mismatches: 0
Token mismatches: 0
Total samples: 12
